In [ ]:
# ========================================
#  Higher Education Students Performance – CatBoostClassifier model, Particle Swarm Optimization, and SHAP
#  Predictive and Interpretable Machine Learning Model for Modeling Students' Final Grade
# ========================================
"""
This research aims to share insights and knowledge regarding the development of a
predictive and interpretable machine learning model, CatBoostClassifier,
for predicting students' end-of-term final grades and
explaining the model’s predictions employing SHAP, an explainable AI method.
As well as share the process of tuning CatBoostClassifier hyperparameters using
Particle Swarm Optimization (PSO), a population-based metaheuristic algorithm,
with 3-fold stratified cross-validation.

Dataset: The dataset used in this study is the "Higher Education Students
Performance Evaluation" dataset, collected from the Faculty of Engineering and
the Faculty of Educational Sciences students in 2019.
Source: Yilmaz, N. & Şekeroğlu, B. (2019). Higher Education Students Performance
Evaluation [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C51G82.
The dataset is a 145 x 33 table of categorical/integer data, where each column
corresponds to a question or attribute, and each row corresponds to a student.
Questions 1-10 are personal questions, questions 11-16 are family questions,
and questions 17-30 cover the student's education habits.
DATA.csv does not contain any helper columns that need to be dropped or broken down into multiple columns.
The target column is GRADE, the student's end-of-term output grade
(0: Fail, 1: DD, 2: DC, 3: CC, 4: CB, 5: BB, 6: BA, 7: AA), and all remaining
columns (personal questions, family questions, education habit questions, and
the STUDENT ID/COURSE ID identifier columns) are used as categorical features.

Symmetries & Asymmetries: Although the questionnaire exhibited structural symmetry in questionnaire design
through questions with a predefined set of answers, the response distribution for
question GRADE showed strong asymmetry, with the highest-engagement
category '1' being 4.375 times more than the lowest-engagement category '0'.
This was heavily imbalanced, which resulted in biased predictions.
The distribution of the GRADE target column is checked
below via a value count. Class imbalance across the eight grade categories is
expected, since some grades are naturally awarded far more often than others,
which can result in biased predictions if left unaddressed.

All stages of the study, including data loading, preprocessing,
model optimization, training, evaluation metrics, and interpretability,
were implemented in Python and executed in the Google Colab environment.
"""



#-----------------------------------------------------Installing & importing libraries
!pip install --quiet catboost pyswarms joblib
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, log_loss, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive
import matplotlib.pyplot as plt
import pyswarms as ps
from joblib import Parallel, delayed



#-----------------------------------------------------
# PREPROCESSING
#-----------------------------------------------------
print("\n\n============ PREPROCESSING... ============")

"""The dataset was read from the CSV file and stored as a DataFrame object (df)."""
drive.mount("/content/gdrive")
df = pd.read_csv("/content/gdrive/MyDrive/DATA.csv")
"""DATA.csv does not contain any helper columns, so no columns were dropped."""
df_clean = df.copy()
print("No helper columns to drop")
"""The categorical target was defined as GRADE, while all remaining columns as categorical features."""
target_col = 'GRADE'
print("Target column:", target_col)
#This will show True in any column where hidden whitespace exists:
#(df_clean.astype(str) != df_clean.astype(str).apply(lambda col: col.str.strip())).any()
#if everything is false:
X = df_clean.drop(columns=[target_col]).astype(str)  #features
y = df_clean[target_col].astype(str)                 #target
#if something is true:
#X = df_clean.drop(columns=[target_col]).astype(str).applymap(lambda v: v.strip())
#y = df_clean[target_col].astype(str).apply(lambda v: v.strip())
"""
The categorical target values were encoded using LabelEncoder to transform class labels into
integer labels suitable for multi-class classification.
The original class labels were preserved for prediction inter-pretation and evaluation metrics.
"""
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_labels = list(le.classes_)
print("Encoded classes (model order):", class_labels)
feature_names = list(X.columns)
print("Feature names:", feature_names)
cat_features = list(range(X.shape[1]))
print("Categorical features:", cat_features)
"""
A target value count was performed, and the presence of a large class imbalance was determined.
In order to mitigate the effects of target class imbalance, balanced class weights were computed
(and they are incorporated into the CatBoost training process
to ensure the model learns from all classes fairly.)
"""
print("Balansirano: najveca klasa <= 2-2,5 puta najmanje klase; klase imaju slican broj primera")
print("Umereno balansirano: najveca klasa > 3-4 puta najmanje klase")
print("Veoma nebalansirano: najveca klasa === 5-10 puta najmanje klase; klasa ima < 20-30 primera")
print("balanced classes?: \n{}\n".format(df_clean[target_col].value_counts()))
# '1' majority
# '0' minority
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_enc),
    y=y_enc
)
print("Class weights:", class_weights)

In [ ]:
#-----------------------------------------------------
# METAHEURISTIC OPTIMIZATION
# Particle Swarm Optimization (PSO) was employed to identify an optimal learning rate, depth, and L2 regularization.
#-----------------------------------------------------
print("\n\n============ Particle Swarm Optimization (PSO)... ============")

"""
The custom parallelized fitness function simultaneously evaluated
all hyperparameter sets proposed by the optimizer and returned their performance scores.
The custom parallelized fitness function evaluates hyperparameters by
training temporary CatBoost models using stratified cross-validation.

To ensure reproducibility, a fixed random seed was used throughout all hyperparameter evaluations.

Stratified K-fold cross-validation with three folds was applied
to enforce symmetry in the target class distribution across folds.
The 3-fold setup provided a balance between computational efficiency and statistical reliability,
whereas enabling shuffling avoided ordering bias in the student data.
With cross-validated model training, overfitting was reduced to a single train/test split,
and each particle was evaluated using 3-fold stratified cross-validation.
"""
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)  # smaller for speed
def pso_objective_parallel(particles):
    def evaluate_particle(p):
        lr = float(p[0])
        depth = int(round(p[1]))
        l2 = int(round(p[2]))
        random_strength = float(p[3])
        fold_scores = []

        for train_idx, test_idx in kf.split(X, y_enc):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y_enc[train_idx], y_enc[test_idx]

            """
            CatBoost’s optimized data structure, Pool,
            was used to store features, labels, and cate-gorical features.
            """
            tr_pool = Pool(X_tr, y_tr, cat_features=cat_features)
            te_pool = Pool(X_te, y_te, cat_features=cat_features)

            """
            Temporary CatBoostClassifier models were configured with
            PSO-tuned learning_rate, depth, l2_leaf_reg, and random_strength,
            500 iterations, ‘MultiClass’ loss_function,
            ‘Mul-tiClass’ eval_metric, fixed random_seed,
            previously calculated balanced class_weights,
            and 50 early_stopping_rounds.
            random_strength adds randomness to split selection, which acts as extra
            regularization and helps the search find hyperparameters that resist
            overfitting on this small, noisy dataset.
            """
            model = CatBoostClassifier(
                iterations=500,  # smaller for speed
                learning_rate=lr,
                depth=depth,
                l2_leaf_reg=l2,
                random_strength=random_strength,
                loss_function='MultiClass',
                eval_metric='MultiClass',
                random_seed=RANDOM_STATE,
                early_stopping_rounds=50,
                verbose=False,
                class_weights=class_weights,
                thread_count=-1   # use all cores
            )
            model.fit(tr_pool, eval_set=te_pool, use_best_model=True)
            preds = np.argmax(model.predict_proba(te_pool), axis=1)
            """
            The performance was measured using the mean macro F1-score,
            suitable for class imbalance because it treats all classes equally.
            """
            fold_scores.append(f1_score(y_te, preds, average='macro'))

        """Minimizing the mean macro F1-score across folds resulted in max-imizing Macro F1."""
        return -np.mean(fold_scores)

    # Parallel evaluation of all particles
    return np.array(Parallel(n_jobs=-1)(delayed(evaluate_particle)(p) for p in particles))

"""PSO search space was defined in the code as parameter bounds
from 0.01 to 0.10 for learning rate,
from 3 to 10 for depth,
from 1 to 10 for L2 regularization,
from 0 to 5 for random_strength (added regularization dimension)
"""
bounds = (
    np.array([0.01, 3, 1, 0.0]),
    np.array([0.10, 10, 10, 5.0])
)
"""
*The optimizer was implemented using the Global Best topology
with 10 particles, 4 dimensions, defined parameter bounds, and set options.
*The custom parallelized fitness function^^^ simultaneously evaluated
all hyperparameter sets proposed by the optimizer and returned their performance scores.
The optimizer relied on those performance scores for returning the
optimal-performing hyperparameter combination after 15 iterations.
*The number of particles and iterations was chosen to balance optimization
quality and computational efficiency.
"""
optimizer = ps.single.GlobalBestPSO(
    n_particles=10,
    dimensions=4,
    options={'c1':1.4, 'c2':1.4, 'w':0.7},
    bounds=bounds
)
cost, pos = optimizer.optimize(pso_objective_parallel, iters=15)  # more thorough search
best_params = {
    'learning_rate': float(pos[0]),
    'depth': int(round(pos[1])),
    'l2_leaf_reg': int(round(pos[2])),
    'random_strength': float(pos[3])
}
print("\nBest parameters found by PSO after 15 iterations:")
print(best_params)

In [ ]:
#-----------------------------------------------------
# Oprimized CatBoostClassifier model
#-----------------------------------------------------
print("\n\n============ Oprimized CatBoostClassifier model... ============")

"""For the final CatBoostClassifier model evaluation with optimized hyperparameters,
the dataset was split into 80% training and 20% testing, fixed random state,
and stratified sampling to ensure the class distribution is the same in train and test splits,
which is important for classification reliability."""
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=RANDOM_STATE
)
"""
In this study, five CatBoostClassifier models were independently trained with
a different random seed and identical optimized hyperparameters,
which ensures performance differences came only from randomness.
"""
models = []
for seed in [0, 1, 2, 3, 4]:
    m = CatBoostClassifier(
        iterations=2000,
        learning_rate=best_params['learning_rate'],
        depth=best_params['depth'],
        l2_leaf_reg=best_params['l2_leaf_reg'],
        random_strength=best_params['random_strength'],
        loss_function='MultiClass',
        eval_metric='MultiClass',
        random_seed=seed,
        early_stopping_rounds=100,
        verbose=False,
        class_weights=class_weights
    )
    m.fit(
        Pool(X_train, y_train, cat_features=cat_features),
        eval_set=Pool(X_test, y_test, cat_features=cat_features),
        use_best_model=True
    )
    models.append(m)

"""
Final predictions were obtained with a soft voting approach by averaging
class probabilities across models, which reduced prediction variance and
improved robustness, particularly for minority classes.
"""
probs = np.mean( #soft voting (%,%,%,%,%,%,%,%)
    [m.predict_proba(Pool(X_test, y_test, cat_features=cat_features)) for m in models],
    axis=0
)
y_pred = np.argmax(probs, axis=1)#averaging
"""
A function was defined to display class prediction
probabilities of the first test sample
as a percentage for individual samples.
"""
def pretty_probs(probs, labels):
    return {labels[i]: f"{round(100*probs[i],2)}%" for i in range(len(labels))}
print("\nExample probabilities for first test sample:")
print(pretty_probs(probs[0], class_labels))








#-----------------------------------------------------
# Evaluation metrics
#-----------------------------------------------------
print("\n\n============ Evaluation metrics... ============")
acc = accuracy_score(y_test, y_pred)
ll = log_loss(y_test, probs)
macro_f1 = f1_score(y_test, y_pred, average='macro')
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Accuracy: {acc:.4f}")
print(f"Log-loss: {ll:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall   : {macro_recall:.4f}")
#Kako da znaš da li balansacija menja performanse modela?
#A) Pogledaj classification report po klasama:
# Vidi se visoka preciznost za najbrojniju GRADE klasu
# Vidi se niska preciznost/recall za retke GRADE klase
# === To znaci da se model "oslanja" na najbrojniju klasu i da bi balansacija pomogla
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_labels))
#B) Pogledaj konfuzionu matricu
# Ako vidiš da model sve gura u jednu klasu → nebalansirano
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
print(confusion_matrix(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
#-----------------------------------------------------
# SHAP-based feature importance
#-----------------------------------------------------
print("\n\n============ SHAP... ============")

"""
SHAP values and CatBoost’s built-in feature importance were
used to interpret the CatBoostClassifier model predictions.
Because SHAP is model-specific, the first model was used as a representative model.
The training data was converted into Pool format and used for computing SHAP values.
The mean absolute SHAP values were calculated across all samples and
classes to obtain global feature importance scores.
These values were compared with CatBoost’s built-in feature importance to ensure con-sistency.
Multi-class output was handled by producing one global importance value per feature.
The feature importance table was created with feature names, SHAP im-portance,
and CatBoost’s built-in feature importance, displaying all questions.
The table was sorted by SHAP importance.
"""
model = models[0]
train_pool = Pool(X_train, y_train, cat_features=cat_features)
shap_vals = model.get_feature_importance(train_pool, type='ShapValues')
shap_vals_no_base = shap_vals[:, :, :-1]  # remove baseline
mean_shap = np.mean(shap_vals_no_base, axis=(0,1))
mean_abs_shap = np.mean(np.abs(shap_vals_no_base), axis=(0,1))

cb_raw = np.array(model.get_feature_importance(train_pool, type='FeatureImportance'))
n_features = len(feature_names)
if len(cb_raw) != n_features:
    n_classes = len(cb_raw) // n_features
    cb_importance = cb_raw.reshape(n_classes, n_features).sum(axis=0)
else:
    cb_importance = cb_raw

fi_df_mas = pd.DataFrame({
    'feature': feature_names,
    'mean_shap': mean_shap,
    'mean_abs_shap': mean_abs_shap,
    'cb_importance': cb_importance
}).sort_values('mean_abs_shap', ascending=False)

fi_df_ms = pd.DataFrame({
    'feature': feature_names,
    'mean_shap': mean_shap,
    'mean_abs_shap': mean_abs_shap,
    'cb_importance': cb_importance
}).sort_values('mean_shap', ascending=False)

fi_df_cb = pd.DataFrame({
    'feature': feature_names,
    'mean_shap': mean_shap,
    'mean_abs_shap': mean_abs_shap,
    'cb_importance': cb_importance
}).sort_values('cb_importance', ascending=False)

print(f"\nAll features influencing {target_col} - sort(mas):")
print(fi_df_mas)
print(f"\nAll features influencing {target_col} - sort(ms):")
print(fi_df_ms)
print(f"\nAll features influencing {target_col} - sort(cb):")
print(fi_df_cb)

#mas
plt.figure(figsize=(8,10))
plt.barh(fi_df_mas.feature, fi_df_mas.mean_abs_shap, alpha=0.6, label='Mean |SHAP|')
plt.barh(fi_df_mas.feature, fi_df_mas.mean_shap, alpha=0.6, label='Mean SHAP')
plt.barh(fi_df_mas.feature, fi_df_mas.cb_importance, alpha=0.4, label='CatBoost Importance')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()
#ma
plt.figure(figsize=(8,10))
plt.barh(fi_df_ms.feature, fi_df_ms.mean_abs_shap, alpha=0.6, label='Mean |SHAP|')
plt.barh(fi_df_ms.feature, fi_df_ms.mean_shap, alpha=0.6, label='Mean SHAP')
plt.barh(fi_df_ms.feature, fi_df_ms.cb_importance, alpha=0.4, label='CatBoost Importance')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()
#cb
plt.figure(figsize=(8,10))
plt.barh(fi_df_cb.feature, fi_df_cb.mean_abs_shap, alpha=0.6, label='Mean |SHAP|')
plt.barh(fi_df_cb.feature, fi_df_cb.mean_shap, alpha=0.6, label='Mean SHAP')
plt.barh(fi_df_cb.feature, fi_df_cb.cb_importance, alpha=0.4, label='CatBoost Importance')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()

#-----separate----
plt.figure(figsize=(8,10))
plt.barh(fi_df_mas.feature, fi_df_mas.mean_abs_shap, alpha=0.6, label='Mean |SHAP|')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,10))
plt.barh(fi_df_ms.feature, fi_df_ms.mean_shap, alpha=0.6, label='Mean SHAP')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,10))
plt.barh(fi_df_cb.feature, fi_df_cb.cb_importance, alpha=0.4, label='CatBoost Importance')
plt.gca().invert_yaxis()
plt.xlabel("Feature Importance")
plt.title(f"Influential Features for {target_col}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#To understand how weekly study hours (column "17") influence predictions:

import shap
class_idx = 1  # positive class
shap_class = shap_vals_no_base[:, class_idx, :]  # (n_samples, n_features)
shap.dependence_plot(
    "17",
    shap_class,
    X_train,
    feature_names=feature_names
)

#or check distribution:
study_hours_idx = feature_names.index("17")
plt.hist(shap_class[:, study_hours_idx], bins=50)
plt.title("SHAP value distribution for weekly study hours - column 17 (class 1)")
plt.xlabel("SHAP value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
"""
 The default CatBoostClassifier model, without hyperparameter tuning or class re-balancing,
 was run to establish a baseline for PSO and optimized model evaluation.
 The model was trained using the same data split and metrics.
 It uses the same Pool/cat_features setup as the optimized model so that categorical
 encoding is handled identically and the only real differences between baseline and
 optimized model are PSO tuning, class balancing, and the 5-seed ensemble.
"""
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, log_loss, f1_score, precision_score, recall_score, classification_report, confusion_matrix)

# ---------------------------
# Default CatBoost Baseline
# ---------------------------
baseline_model = CatBoostClassifier(
    random_state=42,
    verbose=0
)
baseline_model.fit(Pool(X_train, y_train, cat_features=cat_features)) # Train
baseline_test_pool = Pool(X_test, y_test, cat_features=cat_features)
y_pred = baseline_model.predict(baseline_test_pool) # Predict
baseline_probs = baseline_model.predict_proba(baseline_test_pool) # needed for its own log-loss

# ---------------------------
# Evaluation Metrics
# ---------------------------
print("\n=== Default CatBoost Baseline ===")
acc = accuracy_score(y_test, y_pred)
ll = log_loss(y_test, baseline_probs)
macro_f1 = f1_score(y_test, y_pred, average="macro")
macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Accuracy: {acc:.4f}")
print(f"***Log-loss: {ll:.4f}")
print(f"Macro-F1       : {macro_f1:.4f}")
print(f"Macro-Precision: {macro_precision:.4f}")
print(f"Macro-Recall   : {macro_recall:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

print("\n=== Confusion Matrix: ===")
# Ako vidiš da model sve gura u jednu klasu → nebalansirano
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
print(confusion_matrix(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
#xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx Data verification from AI project
#DataFrame object
print(type(df))
print("Loaded:", df.shape)
print("Columns:", list(df.columns))
#uzorak data seta:
print("head: \n{}\n".format(df_clean.head())) #print(df_clean[0:5]) #ispisi prvih 5 redova iz dataframe-a
print("tail: \n{}\n".format(df_clean.tail)) #ispisivanje poslednjih 5 redova
#indeksi pocinju od 0, korak je 1,
print("index: {}\n".format(df_clean.index))
print("1. red dataframe-a: \n{}\n".format(df_clean.iloc[0])) #1. red dataframe-a
#dimenzija dataframe-a (2D):
print("DataFrame ima %d dimenzije" % df_clean.ndim)
#ispisi featur-e/kolone, target-e/labele, vrednosti
print("Featurs/Kolone: ", df_clean.columns)
print(f"Odgovori za {target_col} (target/label variable): ", df_clean[target_col].unique())
print("Vrednosti: \n", df_clean.values) #prosledjuje samo vrednosti
#koliko ima redova i kolona
(row, col) = df_clean.shape
print("Dataframe ima %d zapisa/redova i %d obelezja/kolona(labels)\n" % (row, col))
print("Dataframe ima %d zapisa/redova i %d obelezja/kolona(labels)\n" % (len(df_clean), len(df_clean.columns)))
#transponovan dataframe
print("Transponovan dataframe: \n{}".format(df_clean.T))
#sve max i min vrednosti kolona
import numpy as np
max = df_clean.apply(np.max)
min = df_clean.apply(np.min)
print("max vrednosti kolona: \n{}\n".format(max))
print("min vrednosti kolona: \n{}\n".format(min))
print(df_clean.info())
print("(.info) --- Zakljucak:\nDataFrame ima %d redova i %d kolona\nVidimo kako su imenovane kolone\nKolone imaju int podatke (int64)\nsvaka kolona ima non-null vrednosti, tj. nemamo nedostajuce vrednosti\n" % (row, col))
print(df_clean.describe()) #osnovne statisticke podatke
print("(.describe) --- Zakljucak:\ncount: broj non-null podataka za svaku kolonu, nemamo prazna polja\nmean(avg): srednja vrednost svakog pitanja\nstd(standardna devijacija): disperzija vrednosti svake kolone oko mean, veci std ukazuje na vecu disperziju oko mean vrednosti\nmin,max: najmanja i najveca vrednost za svaku kolonu\n%: distribucija podataka, npr 25% ispitanika je izabralo odredjenu vrednost za odredjeno pitanje\n")
print(df_clean.corr())
print(f"(.corr) --- Zakljucak:\nprocena stepena linearnog odnosa izmedju 2 numbericke variable\nblizu 1: jaka pozitivna linearna veza, v1 se povecava kako se v2 povecava\nblizu-1: jaka negativna linearna veza, v1 se povecava kako se v2 povecava\nblizu 0: slaba ili nikakva linearna veza\npogledati korelacije kolona sa {target_col}\n")